In [37]:
!tar -xzvf experiments.tar.gz

x experiments/
x experiments/7/
x experiments/7/lstm/
x experiments/7/lstm/length_wise_metrics.tsv
x experiments/7/lstm/hparams.json
x experiments/7/lstm/train_losses.tsv
x experiments/7/lstm/metrics.tsv
x experiments/7/trf/
x experiments/7/trf/length_wise_metrics.tsv
x experiments/7/trf/hparams.json
x experiments/7/trf/train_losses.tsv
x experiments/7/trf/metrics.tsv
x experiments/135/
x experiments/135/lstm/
x experiments/135/lstm/length_wise_metrics.tsv
x experiments/135/lstm/hparams.json
x experiments/135/lstm/train_losses.tsv
x experiments/135/lstm/metrics.tsv
x experiments/135/trf/
x experiments/135/trf/length_wise_metrics.tsv
x experiments/135/trf/hparams.json
x experiments/135/trf/train_losses.tsv
x experiments/135/trf/metrics.tsv
x experiments/47/
x experiments/47/lstm/
x experiments/47/lstm/length_wise_metrics.tsv
x experiments/47/lstm/hparams.json
x experiments/47/lstm/train_losses.tsv
x experiments/47/lstm/metrics.tsv
x experiments/47/trf/
x experiments/47/trf/length_wise_m

In [ ]:
import os
import json
import plotly.express as px
from plotly import graph_objects as go
import numpy as np
import pandas as pd
import glob
import math

In [ ]:
lstm_jsons = glob.glob('experiments/*/lstm/hparams.json')
trf_jsons = glob.glob('experiments/*/trf/hparams.json')

In [ ]:
def fetch_data(
    arch,
    gram,
    vocab_size,
    states,
    get_length_wise=False
):
    
    if arch == 'lstm':
        jsons = lstm_jsons
    elif arch == 'trf':
        jsons = trf_jsons
    elif arch == 'any':
        jsons = lstm_jsons + trf_jsons
    else:
        raise KeyError
    
    if gram not in ('pfsa', 'pcfg', 'any'):
        raise KeyError
    if vocab_size not in (1000, 5000, 'any'):
        raise KeyError
    if states not in (2, 4, 8, 16, 32, 64, 'any'):
        raise KeyError
    
    result = []
    
    for j in jsons:
        data = json.load(open(j))
        if (
            data['grammar_type'] == gram or gram == 'any'
        ) and (
            data['grammar_num_symbols'] == vocab_size or vocab_size == 'any'
        ) and (
            data['grammar_formalism_arg'] == states or states == 'any'
        ):
            if get_length_wise:
                result.append((
                    j.split(os.path.sep)[1],
                    data,
                    pd.read_csv(j.replace('hparams.json', 'metrics.tsv'), sep='\t'),
                    pd.read_csv(j.replace('hparams.json', 'length_wise_metrics.tsv'), sep='\t')
                ))
            else:
                result.append((
                    j.split(os.path.sep)[1],
                    data,
                    pd.read_csv(j.replace('hparams.json', 'metrics.tsv'), sep='\t')
                ))
    
    return result

In [ ]:
def best_step(df: pd.DataFrame, crit):
    col = crit
    if crit in ('spearman_weighted_avg', 'spearman_by_len'):
        return df[df[col] == df[col].max()].iloc[-1]
    elif crit == 'ce':
        return df[df[col] == df[col].min()].iloc[-1]
    else:
        raise KeyError(f'unknown: {crit}')

In [ ]:
df = pd.DataFrame()

for experiment_num, hparams, metrics, lwm in fetch_data('any', 'any', 'any', 'any', True):
    
    best_step_MSLAC = best_step(metrics, 'spearman_weighted_avg')
    best_step_ce = best_step(metrics, 'ce')
    
    best_step_mSLAC = best_step(lwm, 'spearman_by_len')
    
    df = pd.concat((df, pd.DataFrame({
        'Experiment number': [int(experiment_num)],
        'Architecture': [hparams['model_type']],
        'Grammar type': [hparams['grammar_type']],
        'Vocabulary size': [hparams['grammar_num_symbols']],
        'Log number of states or non-terminals': [math.log(hparams['grammar_formalism_arg'], 2)],
        'Seed': [hparams['grammar_seed']],
        'Best step by MSLAC': [best_step_MSLAC['step']],
        'Best step by mSLAC': [best_step_mSLAC['step']],
        'Best step by CE': [best_step_ce['step']],
        'Mean length': [hparams['train_data_stats']['mean_length']],
        'Entropy': [hparams['grammar_actual_entropy']],
        'Best MSLAC': [best_step_MSLAC['spearman_weighted_avg']],
        'Best mSLAC': [best_step_mSLAC['spearman_by_len']],
        'Best CE': [best_step_ce['ce']],
        'Excess entropy': [hparams['train_data_ee']],
        'p-value sum (MSLAC)': [best_step_MSLAC['sum_of_pvals']],
        'Variance': [hparams['var']]
    })), ignore_index=True)
    
df['Token-wise entropy'] = df['Entropy'] / df['Mean length']
df['Best KL divergence'] = df['Best CE'] - df['Token-wise entropy']
    
df = df.sort_values(by=['Experiment number', 'Architecture'])

df = df.set_index('Experiment number')

df

In [ ]:
def subset(data, key_val_op_triples) -> pd.DataFrame:
    copy = data.copy()
    
    for key_val_op in key_val_op_triples:
        key, val, op = key_val_op
        if type(val) == str:
            val = '\'' + val + '\''
        copy = copy[eval(f"copy['{key}'] {op} {val}")]
    
    return copy

In [ ]:
pfsa_corr = subset(df, [('Grammar type', 'pfsa', '==')]).corr(numeric_only=True)
pcfg_corr = subset(df, [('Grammar type', 'pcfg', '==')]).corr(numeric_only=True)

In [ ]:
def plot_regs(data, x, y, title_pref):
    
    corr_x_y = pcfg_corr[x][y] if 'PCFG' in title_pref else pfsa_corr[x][y]
    
    # if abs(corr_x_y) < 0.5:
    #     return
    
    if y == 'Entropy':
        x, y = y, x
    
    x_axis_min = data[x].min()
    y_axis_min = data[y].min()
    x_axis_max = data[x].max()
    y_axis_max = data[y].max()
    x_padding = (x_axis_max - x_axis_min) * 0.05
    y_padding = (y_axis_max - y_axis_min) * 0.05
    x_axis_range = [x_axis_min - x_padding, x_axis_max + x_padding]
    y_axis_range = [y_axis_min - y_padding, y_axis_max + y_padding]

    if 'PCFG' in title_pref:
        colorbar_title = 'Log # Non-terms'
        x_title = x.replace(
            'Log number of states or non-terminals',
            'Log # non-terms'
        )
        y_title = y.replace(
            'Log number of states or non-terminals',
            'Log # non-terms'
        )
    else:
        colorbar_title = 'Log # States'
        x_title = x.replace(
            'Log number of states or non-terminals',
            'Log # states'
        )
        y_title = y.replace(
            'Log number of states or non-terminals',
            'Log # states'
        )

    fig = px.scatter(
        data,
        x,
        y,
        symbol='Architecture',
        size='Vocabulary size',
        text='Variance',
        title=title_pref + f'{y_title} vs. {x_title} (r~{corr_x_y:.2f})',
        color='Log number of states or non-terminals',
        opacity=0.3
    ).update_layout(
        autosize=False,
        width=600,
        height=450,
        margin=dict(l=70, r=120, t=60, b=60),
        xaxis=dict(range=x_axis_range),
        yaxis=dict(range=y_axis_range),
        legend=dict(
            orientation='v',
            yanchor='top',
            y=0.38,         # sits just below where colorbar ends
            xanchor='left',
            x=1.02,
            bgcolor='rgba(255, 255, 255, 0.8)',
            bordercolor='rgba(0, 0, 0, 0.2)',
            borderwidth=1,
            font=dict(size=11),
        ),
        hovermode='closest',
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=13)
    ).update_coloraxes(
        colorscale='RdYlBu_r',
        cmin=data['Log number of states or non-terminals'].min(),
        cmax=data['Log number of states or non-terminals'].max(),
        colorbar=dict(
            title=dict(text=colorbar_title, font=dict(size=12)),
            tickfont=dict(size=10),
            x=1.0,
            y=1.04,
            len=0.6,        # takes up top 60% of plot height
            thickness=15,
            yanchor='top',
        )
    )
    
    if 'CE' in y and 'Token-wise' in x:
        endpoint = max(x_axis_range[1], y_axis_range[1])
        fig.add_shape(
            type='line',
            x0=0, y0=0,
            x1=endpoint, y1=endpoint,
            line=dict(color='gray', width=1.5, dash='dash'),
            layer='below'
        )
        
    fig.add_annotation(
        text="Marker size:",
        xref="paper", yref="paper",
        x=1.02, y=0.05,
        showarrow=False,
        font=dict(size=12, color="gray"),
        xanchor='left'
    )
    fig.add_annotation(
        text="Vocab=1K, 5K",
        xref="paper", yref="paper",
        x=1.02, y=0.0,
        showarrow=False,
        font=dict(size=12, color="gray"),
        xanchor='left'
    )
    
    fig.show()

In [ ]:
done = set()

for i, x in enumerate(pfsa_corr.columns):
    for y in pfsa_corr.columns:
        if (y, x) in done or (x, y) in done:
            continue
        if x == y:
            continue
        if 'Seed' in (x, y) or 'Vocabulary size' in (x, y) or 'p-value' in x or 'p-value' in y:
            continue
        if y == 'Token-wise entropy':
            x, y = y, x
        if y == 'Entropy':
            x, y = y, x
        done.add((x, y))
        done.add((y, x))
        plot_regs(
            subset(
                df,
                [('Grammar type', 'pfsa', '==')]
            ), x, y,
            title_pref = 'PFSAs: '
        )

        plot_regs(
            subset(
                df,
                [('Grammar type', 'pcfg', '==')]
            ), x, y,
            title_pref = 'PCFGs: '
        )

In [31]:
test_df = df[(df['Architecture'] == 'trf') & (df['Grammar type'] == 'pfsa')]

In [34]:
test_df = test_df[(test_df['Seed'] == 0) & (test_df['Vocabulary size'] == 1000)]

In [36]:
test_df[test_df['Variance'] == 1.0]

,Architecture,Grammar type,Vocabulary size,Log number of states or non-terminals,Seed,Best step by MSLAC,Best step by mSLAC,Best step by CE,Mean length,Entropy,Best MSLAC,Best mSLAC,Best CE,Excess entropy,p-value sum (MSLAC),Variance,Token-wise entropy,Best KL divergence
Experiment number,,,,,,,,,,,,,,,,,,
1,trf,pfsa,1000,1.0,0,6000.0,2600.0,5000.0,2.970625,7.506837,0.806175,0.900430,2.238441,5.978066,2.657251e-06,1.0,2.527023,-0.288581
2,trf,pfsa,1000,2.0,0,4700.0,4800.0,5800.0,5.097094,8.016413,0.720852,0.799942,2.084914,8.561621,2.027965e-07,1.0,1.572742,0.512172
3,trf,pfsa,1000,3.0,0,6400.0,8600.0,8500.0,8.937500,8.602941,0.620950,0.683327,1.979069,10.358926,4.221422e-92,1.0,0.962567,1.016502
65,trf,pfsa,1000,1.0,0,5900.0,2000.0,5900.0,2.963406,7.506837,0.803536,0.902809,2.241984,6.016033,3.870648e-07,1.0,2.533178,-0.291195
67,trf,pfsa,1000,2.0,0,4900.0,4000.0,5800.0,5.091281,8.016413,0.717852,0.797254,2.088080,8.561100,6.608598e-07,1.0,1.574537,0.513543
69,trf,pfsa,1000,3.0,0,9400.0,9400.0,10000.0,8.931375,8.602941,0.628419,0.672978,1.975086,10.352198,1.298807e-90,1.0,0.963227,1.011859
72,trf,pfsa,1000,4.0,0,8800.0,10400.0,8800.0,17.156063,9.240372,0.513683,0.565942,1.891738,11.743071,6.936406e-64,1.0,0.538607,1.353132
74,trf,pfsa,1000,5.0,0,8400.0,8400.0,10300.0,32.231469,9.905296,0.401855,0.459840,2.039721,12.874513,1.233793e-22,1.0,0.307318,1.732404
76,trf,pfsa,1000,6.0,0,9000.0,7300.0,9700.0,56.137500,10.581440,0.221648,0.359919,3.106953,13.804068,8.207421e-02,1.0,0.188491,2.918461
